# Resolution Rate by First Error Type
# Taux de Résolution selon le Type de Première Erreur

**🇬🇧** This notebook analyzes how the type of the first error encountered on a problem influences whether the user eventually resolves it. The unit of analysis is the **(user, problem) pair** where the first submission was NOT AC — pairs solved on the first try are excluded entirely.

**Note:** `avg_attempts_to_ac` counts from the first submission (the error itself counts as attempt 1). A value of 2.57 means: 1 failed first submission + 1.57 additional submissions on average to reach AC.

**🇫🇷** Ce notebook analyse comment le type de la première erreur rencontrée influence la probabilité de finalement résoudre un problème. L'unité d'analyse est la **paire (utilisateur, problème)** dont la première soumission n'était pas AC. Les paires réussies du premier coup sont entièrement exclues.

**Note :** `avg_attempts_to_ac` compte depuis la première soumission (l'erreur elle-même compte comme tentative 1). Une valeur de 2.57 signifie : 1 première soumission échouée + 1.57 soumissions supplémentaires en moyenne pour atteindre AC.

> ⚠️ La catégorie **"Other"** (MLE, OLE, Judge Error) est exclue de toutes les visualisations : volume insuffisant (34 à 380 paires selon le niveau) pour être statistiquement fiable.

---

**CSVs utilisés :** `data/processed/atcoder_resolution_by_first_error.csv`, `atcoder_resolution_by_first_error_group.csv`

**Structure :**
- **S1** : Volume de paires (utilisateur, problème) par type de première erreur — combien de chemins débutent par WA, TLE, CE, RE / Volume by first error type
- **S2** : Taux de résolution par type de première erreur × difficulté — quel premier type d'erreur mène le plus souvent à l'AC / Resolution rate by first error × difficulty
- **S3** : Tentatives moyennes avant AC par type × difficulté / Average attempts before AC by first error type × difficulty
- **S4** : Taux de résolution par type d'erreur × groupe pour chaque niveau — heatmaps comparant les groupes G1–G6 / Resolution rate by error type × group per level


In [ ]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

DIFFICULTY_ORDER = ['A', 'B', 'C', 'D', 'E', 'F']
GROUP_ORDER      = ['G1', 'G2', 'G3', 'G4', 'G5', 'G6']
ERROR_ORDER      = ['WA', 'TLE', 'CE', 'RE']  # 'Other' excluded

# Consistent color scheme across all notebooks
# WA=red, TLE=orange, CE=blue, RE=purple
ERR_COLORS  = {'WA': '#e74c3c', 'TLE': '#f39c12', 'CE': '#3498db', 'RE': '#9b59b6'}
ERR_MARKERS = {'WA': 'o', 'TLE': 's', 'CE': '^', 'RE': 'D'}

# Load pipeline outputs — filter out unreliable 'Other' category
df_err = (
    pl.read_csv('../data/processed/atcoder_resolution_by_first_error.csv')
    .filter(pl.col('first_error_type') != 'Other')
)
df_err_group = (
    pl.read_csv('../data/processed/atcoder_resolution_by_first_error_group.csv')
    .filter(pl.col('first_error_type') != 'Other')
)

print('Resolution by first error:        ', df_err.shape)
print('Resolution by first error × group:', df_err_group.shape)
print()
print(df_err.sort(['difficulty', 'first_error_type']))

---
## Section 1 — Volume of Pairs per First Error Type
## Section 1 — Volume de Paires par Type de Première Erreur

**Why this figure:** Resolution rates are meaningless without knowing the volume behind them. This figure shows the absolute number of (user, problem) pairs for each (difficulty, error type) combination. It reveals which error types are most common at each level and which results should be interpreted with caution due to low sample size (e.g. TLE at level A: 958 pairs).

**Pourquoi cette figure :** Les taux de résolution n'ont de sens que si l'on connaît le volume de données derrière eux. Cette figure montre le nombre absolu de paires (utilisateur, problème) pour chaque combinaison (difficulté, type d'erreur). Elle révèle quelles erreurs sont les plus fréquentes à chaque niveau et quels résultats doivent être interprétés avec prudence (ex. TLE au niveau A : 958 paires).

---

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

x = np.arange(len(DIFFICULTY_ORDER))
w = 0.2

for i, err in enumerate(ERROR_ORDER):
    vals = []
    for d in DIFFICULTY_ORDER:
        subset = df_err.filter(
            (pl.col('difficulty') == d) & (pl.col('first_error_type') == err)
        )
        vals.append(subset['n_pairs'].item() if not subset.is_empty() else 0)
    offset = (i - 1.5) * w
    bars = ax.bar(x + offset, vals, w, label=err, color=ERR_COLORS[err], edgecolor='white', linewidth=0.6)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1500,
                f'{v:,.0f}', ha='center', va='bottom', fontsize=7, rotation=90
            )

ax.set_xticks(x)
ax.set_xticklabels(DIFFICULTY_ORDER)
ax.set_xlabel('Difficulty Level / Niveau de difficulté', fontsize=12)
ax.set_ylabel('Number of pairs / Nombre de paires', fontsize=12)
ax.set_title(
    'Number of (User, Problem) Pairs by First Error Type and Difficulty Level\n'
    'Nombre de paires (utilisateur, problème) par type de première erreur et niveau de difficulté',
    fontsize=13, pad=12
)
ax.legend(title='First error / Première erreur', fontsize=10)
plt.tight_layout()
plt.show()

print('Raw counts / Volumes bruts:')
pivot = df_err.pivot(index='difficulty', on='first_error_type', values='n_pairs').sort('difficulty')
print(pivot)

In [ ]:
# Pie charts — distribution of first error types per difficulty level
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, diff in enumerate(DIFFICULTY_ORDER):
    ax = axes[idx]
    vals = []
    labels = []
    colors = []
    for err in ERROR_ORDER:
        subset = df_err.filter(
            (pl.col('difficulty') == diff) & (pl.col('first_error_type') == err)
        )
        if not subset.is_empty():
            n = subset['n_pairs'].item()
            vals.append(n)
            labels.append(err)
            colors.append(ERR_COLORS[err])

    total = sum(vals)
    ax.pie(
        vals,
        labels=labels,
        colors=colors,
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
        textprops={'fontsize': 10}
    )
    ax.set_title(f'Level {diff}  (n = {total:,})', fontsize=12, fontweight='bold')

fig.suptitle(
    'Distribution of First Error Types per Difficulty Level\n'
    'Répartition des types de première erreur par niveau de difficulté',
    fontsize=14, y=1.02
)
plt.tight_layout()
plt.show()

### Analysis / Analyse

**🇫🇷** WA domine à tous les niveaux sans exception, entre 56% et 71% des paires, quelle que soit la difficulté. Ce qui change, c'est la composition du reste : CE représente 26% des paires au niveau A puis s'effondre à 4% dès le niveau D, pendant que TLE fait le chemin inverse, quasi-inexistante au niveau A (0.4%), elle atteint 26% au niveau D. RE reste stable autour de 13–15% à tous les niveaux. Ce glissement quantifie le changement de nature des obstacles : les problèmes faciles piègent sur de la syntaxe (CE), les problèmes difficiles sur de la complexité algorithmique (TLE). À retenir pour la suite : CE devient un événement rare sur les niveaux D–F, ce qui rend ses taux de résolution et de tentatives d'autant plus significatifs à interpréter.

---

**🇬🇧** WA dominates at every level without exception — between 56% and 71% of pairs regardless of difficulty. What changes is the composition of the rest: CE represents 26% of pairs at level A then collapses to 4% from level D onward, while TLE follows the reverse path — near-absent at level A (0.4%), it reaches 26% at level D. RE remains stable at around 13–15% across all levels. This shift quantifies the changing nature of obstacles: easy problems trap users on syntax (CE), hard problems on algorithmic complexity (TLE). Worth keeping in mind for the following figures: CE becomes a rare event at levels D–F — which makes its resolution rates and attempt counts all the more meaningful to interpret.

---
## Section 2 — Resolution Rate by First Error Type × Difficulty
## Section 2 — Taux de Résolution par Type de Première Erreur × Difficulté

**Why this figure:** Given that the user's first submission resulted in a specific error type, how likely are they to eventually reach AC? This heatmap answers that question across all difficulty levels. Darker green = higher probability of eventually resolving the problem.

**Pourquoi cette figure :** Étant donné que la première soumission de l'utilisateur a produit un type d'erreur spécifique, quelle est la probabilité qu'il finisse par obtenir AC ? Cette heatmap répond à cette question sur tous les niveaux de difficulté. Plus vert = plus forte probabilité de finalement résoudre le problème.

---

In [ ]:
# Build matrix: rows = difficulty, columns = error type
res_matrix = pd.DataFrame(index=DIFFICULTY_ORDER, columns=ERROR_ORDER, dtype=float)
for diff in DIFFICULTY_ORDER:
    for err in ERROR_ORDER:
        subset = df_err.filter(
            (pl.col('difficulty') == diff) & (pl.col('first_error_type') == err)
        )
        res_matrix.loc[diff, err] = subset['resolution_rate'].item() if not subset.is_empty() else np.nan
res_matrix = res_matrix.astype(float)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    res_matrix,
    annot=True, fmt='.1f', cmap='Greens',
    vmin=60, vmax=95,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Resolution rate (%)'},
    ax=ax
)
ax.set_title(
    'Resolution Rate (%) by First Error Type × Difficulty Level\n'
    'Taux de résolution (%) par type de première erreur × niveau de difficulté',
    fontsize=13, pad=12
)
ax.set_xlabel('First Error Type / Type de première erreur', fontsize=12)
ax.set_ylabel('Difficulty Level / Niveau de difficulté', fontsize=12)
plt.tight_layout()
plt.show()

### Analysis / Analyse

**🇫🇷** CE présente le meilleur taux au niveau A (92.2%), puis WA prend le dessus à partir du niveau B et maintient cette position jusqu'au niveau E. TLE est systématiquement le plus bas de A à F, sans exception. À partir du niveau D, l'écart entre WA et TLE se creuse : WA 78.1% contre TLE 66.0% au niveau D, soit 12 pp. Corriger une TLE exige de repenser l'algorithme entièrement, pas seulement de relire un message d'erreur.

Au niveau F, TLE reste clairement en retrait (63.2%). Les trois autres erreurs se resserrent dans un écart de seulement 2.1 pp (RE 69.9%, WA 68.7%, CE 67.8%) : leur ordre n'est pas interprétable à cette échelle.

---

**🇬🇧** CE shows the best resolution rate at level A (92.2%), then WA takes over from level B and holds that position through level E. TLE is consistently the lowest from A to F, without exception. From level D, the gap between WA and TLE widens: WA 78.1% vs TLE 66.0% at level D — a 12 pp difference. Fixing a TLE requires a full algorithmic rethink, not just reading an error message.

At level F, TLE remains clearly the worst (63.2%). The other three error types cluster within a 2.1 pp range (RE 69.9%, WA 68.7%, CE 67.8%): their ordering is not interpretable at this scale.

---
## Section 3 — Average Attempts to AC by First Error Type × Difficulty
## Section 3 — Tentatives Moyennes avant AC par Type de Première Erreur × Difficulté

**Why this figure:** Among pairs that were eventually resolved, how many submissions did it take? This complements the resolution rate: an error type might have a high resolution rate but require many attempts, or vice versa. Note: the count includes the first failed submission (attempt 1 = the initial error).

**Pourquoi cette figure :** Parmi les paires finalement résolues, combien de soumissions ont-elles nécessité ? Cette figure complète le taux de résolution : un type d'erreur peut avoir un bon taux de résolution mais demander beaucoup d'itérations, ou l'inverse. Note : le comptage inclut la première soumission échouée (tentative 1 = l'erreur initiale).

---

In [ ]:
# Build matrix: rows = difficulty, columns = error type
avg_matrix = pd.DataFrame(index=DIFFICULTY_ORDER, columns=ERROR_ORDER, dtype=float)
for diff in DIFFICULTY_ORDER:
    for err in ERROR_ORDER:
        subset = df_err.filter(
            (pl.col('difficulty') == diff) & (pl.col('first_error_type') == err)
        )
        avg_matrix.loc[diff, err] = subset['avg_attempts_to_ac'].item() if not subset.is_empty() else np.nan
avg_matrix = avg_matrix.astype(float)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    avg_matrix,
    annot=True, fmt='.2f', cmap='Oranges',
    vmin=avg_matrix.min().min(), vmax=avg_matrix.max().max(),
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Avg attempts to first AC (includes first failed submission)'},
    ax=ax
)
ax.set_title(
    'Average Attempts to First AC by First Error Type × Difficulty (resolved pairs only)\n'
    'Tentatives moyennes avant le premier AC par type de première erreur × difficulté (paires résolues)',
    fontsize=13, pad=12
)
ax.set_xlabel('First Error Type / Type de première erreur', fontsize=12)
ax.set_ylabel('Difficulty Level / Niveau de difficulté', fontsize=12)
plt.tight_layout()
plt.show()

### Analysis / Analyse

**🇫🇷** WA est l'erreur la plus rapide à corriger à chaque niveau de difficulté, sans exception. C'est le seul résultat parfaitement stable sur l'ensemble du tableau.

RE montre la dégradation la plus forte avec la difficulté : proche de WA au niveau A (2.81 vs 2.57), il devient le plus lent de tous à partir du niveau C et atteint 4.46 au niveau E, le maximum de tout le tableau. C'est le seul vrai renversement de position clair et significatif dans ces données.

---

**🇬🇧** WA is the fastest error to fix at every difficulty level, without exception. It is the only perfectly stable result across the entire table.

RE shows the sharpest degradation with difficulty: close to WA at level A (2.81 vs 2.57), it becomes the slowest of all from level C onward and peaks at 4.46 at level E — the maximum of the entire table. It is the only clear and significant positional reversal in these data.

---
## Section 4 — Resolution Rate by Error Type × Group for Each Difficulty Level
## Section 4 — Taux de Résolution par Type d'Erreur × Groupe pour Chaque Niveau de Difficulté

**Why this figure:** Six heatmaps — one per difficulty level — each showing how resolution rate varies by first error type (rows) and proficiency group (columns). This avoids arbitrarily fixing a single difficulty and reveals whether the patterns observed at level D hold across all levels. Masked cells (grey) correspond to groups that cannot resolve this difficulty by definition — their 0% is a classification artefact, not a recovery difficulty measure.

**Pourquoi cette figure :** Six heatmaps — une par niveau de difficulté — montrant chacune comment le taux de résolution varie selon le type de première erreur (lignes) et le groupe de proficience (colonnes). Cette approche évite de fixer arbitrairement un seul niveau et révèle si les tendances observées au niveau D se retrouvent partout. Les cellules masquées (gris) correspondent aux groupes qui ne peuvent pas résoudre ce niveau par définition — leur 0% est un artefact de classification, pas une mesure de difficulté de récupération.

---

In [ ]:
def build_group_err_matrix(difficulty, metric):
    """
    Build a matrix (error_type × group) for a given difficulty level.
    Cells where resolution_rate == 0 and avg_attempts is null are masked (NaN):
    these groups cannot resolve this difficulty by definition.
    """
    mat = pd.DataFrame(index=ERROR_ORDER, columns=GROUP_ORDER, dtype=float)
    for err in ERROR_ORDER:
        for g in GROUP_ORDER:
            subset = df_err_group.filter(
                (pl.col('difficulty') == difficulty) &
                (pl.col('first_error_type') == err) &
                (pl.col('proficiency_group') == g)
            )
            if subset.is_empty():
                mat.loc[err, g] = np.nan
            else:
                res = subset['resolution_rate'].item()
                avg = subset['avg_attempts_to_ac'].item()
                mat.loc[err, g] = np.nan if (res == 0.0 and avg is None) else subset[metric].item()
    return mat.astype(float)


fig, axes = plt.subplots(2, 3, figsize=(22, 11))
axes = axes.flatten()

for idx, diff in enumerate(DIFFICULTY_ORDER):
    mat = build_group_err_matrix(diff, 'resolution_rate')
    ax = axes[idx]

    # Custom annotations: hide NaN cells
    annot = mat.copy().astype(object)
    for r in mat.index:
        for c in mat.columns:
            annot.loc[r, c] = '' if np.isnan(mat.loc[r, c]) else f'{mat.loc[r, c]:.1f}'

    sns.heatmap(
        mat,
        annot=annot, fmt='',
        cmap='Greens',
        vmin=40, vmax=100,
        linewidths=0.5, linecolor='white',
        mask=mat.isna(),
        cbar=idx == 2,  # show colorbar only once
        cbar_kws={'label': 'Resolution rate (%)'},
        ax=ax
    )
    ax.set_title(f'Level {diff}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Proficiency Group', fontsize=10)
    ax.set_ylabel('First Error Type', fontsize=10)
    ax.tick_params(axis='both', labelsize=9)

fig.suptitle(
    'Resolution Rate (%) by First Error Type × Proficiency Group — per Difficulty Level\n'
    'Taux de résolution (%) par type de première erreur × groupe — par niveau de difficulté',
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.show()

### Analysis / Analyse

**🇫🇷** Le résultat central de ce notebook : **plus l'utilisateur est expert, moins le type de première erreur compte**. Au niveau D, l'écart entre la meilleure et la pire erreur est de 18 pp pour G4, 8 pp pour G5, et seulement 3.5 pp pour G6. TLE et RE, les deux erreurs les plus pénalisantes en taux de résolution (figure 2) et en nombre de tentatives (figure 3), sont précisément les erreurs pour lesquelles l'écart entre groupes est le plus marqué : G4/D/TLE = 49.7%, G4/D/RE = 50.8%. Une fois sur deux, un utilisateur intermédiaire confronté à une TLE ou RE au niveau D abandonne. Ce n'est pas le cas de G6 au niveau F : peu importe l'erreur initiale, G6 résout à plus de 75%, WA 76.6%, TLE 75.2%, CE 78.7%, RE 79.1%. La capacité à surmonter TLE et RE sur des problèmes difficiles est précisément ce qui distingue les experts des groupes intermédiaires. On retrouve la même logique pour G5 sur les problèmes E et G4 sur les problèmes D : les taux sont nettement plus bas, et TLE/RE y marquent une rupture franche. Ces tendances sont cohérentes avec les figures 2 et 3 : TLE et RE sont les erreurs les plus longues et les plus difficiles à corriger, et la figure 4 montre que cette difficulté est avant tout subie par les groupes qui n'ont pas encore les outils algorithmiques pour y répondre.

---

**🇬🇧** The central finding of this notebook: **the more expert the user, the less the type of first error matters**. At level D, the gap between the best and worst error type is 18 pp for G4, 8 pp for G5, and only 3.5 pp for G6. TLE and RE — the two most penalising errors in resolution rate (figure 2) and attempt count (figure 3) — are precisely the errors where the gap between groups is most pronounced: G4/D/TLE = 49.7%, G4/D/RE = 50.8%. One in two intermediate users facing a TLE or RE at level D gives up. The opposite holds for G6 at level F: regardless of the initial error, G6 resolves at over 75% — WA 76.6%, TLE 75.2%, CE 78.7%, RE 79.1%. The ability to overcome TLE and RE on hard problems is exactly what distinguishes expert groups from intermediate ones. The same logic applies to G5 on level-E problems and G4 on level-D problems: rates are markedly lower, and TLE/RE create a clear break. These trends are consistent with figures 2 and 3 — TLE and RE are the hardest and slowest errors to fix — and figure 4 shows that this difficulty is primarily felt by groups who do not yet have the algorithmic tools to address it.